In [1]:
import pickle, random, time
from pathlib import Path

import pandas as pd
import networkx as nx

REPO_ROOT = Path.cwd().parent
PROCESSED = REPO_ROOT / "data" / "processed"
NULLS     = PROCESSED / "nulls"
RESULTS   = REPO_ROOT / "results"
NULLS.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: d:\ITU\2026-NetworkScienceProject


In [2]:
G = pickle.load(open(PROCESSED / "nordic_graph.pkl", "rb"))
print(f"real graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

caps = [d["capacity"] for _, _, d in G.edges(data=True)]
nswap_target = 10 * G.number_of_edges()

real graph: 1763 nodes, 2253 edges


In [3]:
def null_model(G, caps, seed):
    R = nx.Graph(G.edges())
    nx.set_node_attributes(R, dict(G.nodes(data=True)))
    nx.connected_double_edge_swap(R, nswap=nswap_target, seed=seed)
    shuffled = caps.copy()
    random.Random(seed).shuffle(shuffled)
    for (u, v), c in zip(R.edges(), shuffled):
        R[u][v].update(capacity=c, cost=1 / c)
    return R

In [4]:
t0 = time.time()
for seed in range(30):
    R = null_model(G, caps, seed)
    out = NULLS / f"null_{seed:02d}.pkl"
    with open(out, "wb") as f:
        pickle.dump(R, f)
    elapsed = time.time() - t0
    print(f"seed {seed:02d}: {R.number_of_nodes()} nodes, "
          f"{R.number_of_edges()} edges, "
          f"connected={nx.is_connected(R)}, "
          f"{elapsed:.1f}s total")

print(f"\ndone: 30 nulls in {NULLS}")

seed 00: 1763 nodes, 2253 edges, connected=True, 2.2s total
seed 01: 1763 nodes, 2253 edges, connected=True, 4.2s total
seed 02: 1763 nodes, 2253 edges, connected=True, 6.4s total
seed 03: 1763 nodes, 2253 edges, connected=True, 8.6s total
seed 04: 1763 nodes, 2253 edges, connected=True, 10.7s total
seed 05: 1763 nodes, 2253 edges, connected=True, 12.8s total
seed 06: 1763 nodes, 2253 edges, connected=True, 15.1s total
seed 07: 1763 nodes, 2253 edges, connected=True, 17.5s total
seed 08: 1763 nodes, 2253 edges, connected=True, 19.6s total
seed 09: 1763 nodes, 2253 edges, connected=True, 21.8s total
seed 10: 1763 nodes, 2253 edges, connected=True, 23.9s total
seed 11: 1763 nodes, 2253 edges, connected=True, 26.3s total
seed 12: 1763 nodes, 2253 edges, connected=True, 28.7s total
seed 13: 1763 nodes, 2253 edges, connected=True, 31.0s total
seed 14: 1763 nodes, 2253 edges, connected=True, 33.2s total
seed 15: 1763 nodes, 2253 edges, connected=True, 35.4s total
seed 16: 1763 nodes, 2253 ed

In [5]:
real_deg  = sorted(d for _, d in G.degree())
real_caps = sorted(d["capacity"] for _, _, d in G.edges(data=True))

rows = []
for seed in range(30):
    R = pickle.load(open(NULLS / f"null_{seed:02d}.pkl", "rb"))
    rows.append({
        "seed": seed,
        "nodes": R.number_of_nodes(),
        "edges": R.number_of_edges(),
        "degrees_preserved": sorted(d for _, d in R.degree()) == real_deg,
        "capacities_preserved": sorted(d["capacity"] for _, _, d in R.edges(data=True)) == real_caps,
        "connected": nx.is_connected(R),
    })

check = pd.DataFrame(rows)
check.to_csv(RESULTS / "null_check.csv", index=False)
print(f"saved {RESULTS / 'null_check.csv'}")

assert check["degrees_preserved"].all()
assert check["capacities_preserved"].all()
assert check["connected"].all()
print("all 30 nulls verified: degrees, capacities, connectivity")

saved d:\ITU\2026-NetworkScienceProject\results\null_check.csv
all 30 nulls verified: degrees, capacities, connectivity
